# Python scheduler tutorial

This notebook introduces the dependency-light Python scheduler facade. It runs from the repository checkout, does not use network access, and does not require native FFI wheels.

![Scheduler timeline](figures/scheduler-timeline.svg)

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
python_binding = repo_root / "bindings" / "python"
if str(python_binding) not in sys.path:
    sys.path.insert(0, str(python_binding))

import kairo_ecs

kairo_ecs.self_check()

## Schedule deterministic events

The scheduler orders events by simulation tick, then priority, then insertion sequence. Lower priority values dispatch first when events share the same tick.

In [ ]:
scheduler = kairo_ecs.Scheduler()
scheduler.schedule_at(10, priority=2, kind=3)
scheduler.schedule_at(5, priority=9, kind=1)
scheduler.schedule_at(10, priority=1, kind=2)
scheduler.schedule_at(10, priority=1, kind=4)

dispatched_kinds = []
while True:
    outcome, event = scheduler.step()
    if outcome is kairo_ecs.StepOutcome.EMPTY:
        break
    dispatched_kinds.append(event.kind)

assert dispatched_kinds == [1, 2, 4, 3]
assert scheduler.stats() == {
    "current_time_ticks": 10,
    "pending_events": 0,
    "dispatched_events": 4,
}

dispatched_kinds

## Cancel pending work

Cancellation only succeeds while an event is still pending. Unknown, duplicate, and already-dispatched cancellation attempts return `False`.

In [ ]:
scheduler = kairo_ecs.Scheduler()
first = scheduler.schedule_at(1, kind=10)
cancelled = scheduler.schedule_at(2, kind=20)
third = scheduler.schedule_at(3, kind=30)

assert scheduler.cancel(kairo_ecs.EventId(999, 0)) is False
assert scheduler.cancel(cancelled) is True
assert scheduler.cancel(cancelled) is False

scheduler.run_for(10)
trace = [(event.at.ticks, event.kind) for event in scheduler.trace]

assert trace == [(1, 10), (3, 30)]
assert scheduler.cancel(first) is False
assert scheduler.cancel(third) is False

trace

## Use a small trace table

The trace is plain Python data, so tutorials and quick checks can stay deterministic without optional plotting dependencies.

In [ ]:
trace_rows = [
    {
        "tick": event.at.ticks,
        "kind": event.kind,
        "priority": event.priority,
        "sequence": event.sequence,
    }
    for event in scheduler.trace
]

assert trace_rows == [
    {"tick": 1, "kind": 10, "priority": 0, "sequence": 0},
    {"tick": 3, "kind": 30, "priority": 0, "sequence": 2},
]

trace_rows